In [2]:
import torch 
import numpy as np
from chronos import ChronosPipeline
import yfinance as yf

In [1]:
TICKERS = {
    
    "Amazon": "AMZN",
    "Intel": "INTC",
    "AMD": "AMD",
    "General Electric": "GE",
}

In [3]:
def load_data(company_name, period="10y"):
    ticker = TICKERS[company_name]
    data = yf.download(ticker, period=period)
    data.reset_index(inplace=True)
    return  data
data = load_data("Amazon")
price = data["Close"]

[*********************100%***********************]  1 of 1 completed


In [7]:
def chronos_predict(close_prices, months=6):
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",  # version légère
        device_map="cpu",
        torch_dtype=torch.float32,
    )

    context = torch.tensor(close_prices[-200:], dtype=torch.float32)
    n_days = months * 21

    forecast = pipeline.predict(
        context.unsqueeze(0),
        prediction_length=n_days,
        num_samples=20,
    )

    # Médiane des prédictions
    median_forecast = forecast[0].median(dim=0).values.numpy()
    return median_forecast

predicted_prices = chronos_predict(price.squeeze().to_numpy())
print(predicted_prices)

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


[247.73912 247.73912 249.39082 247.73912 244.43594 242.78445 244.43594
 241.13275 237.82957 236.17809 237.82957 231.2232  234.52638 237.82957
 234.52638 239.48128 237.82957 234.52638 237.82957 232.8749  234.52638
 229.57172 227.92001 227.92001 229.57172 232.8749  236.17809 232.8749
 232.8749  234.52638 234.52638 227.92001 231.2232  229.57172 226.26852
 224.61682 227.92001 222.96535 226.26852 226.26852 227.92001 224.61682
 224.61682 226.26852 227.92001 229.57172 227.92001 229.57172 224.61682
 224.61682 224.61682 227.92001 231.2232  227.92001 229.57172 227.92001
 224.61682 224.61682 226.26852 232.8749  229.57172 229.57172 229.57172
 226.26852 226.26607 227.92989 227.92989 229.5935  224.60245 227.92989
 226.26607 229.5935  229.5935  224.60245 222.93863 224.60245 226.26607
 224.60245 222.93863 222.93863 221.2748  224.60245 224.60245 222.93863
 222.93863 221.2748  226.26607 224.60245 224.60245 227.92989 227.92989
 229.5935  229.5935  227.92989 229.5935  229.5935  229.5935  229.5935
 236.248

In [8]:
np.save("predicted_prices.npy", predicted_prices)